In [1]:
import tensorflow as tf
from tensorflow.keras.datasets import mnist
from tensorflow.keras.applications import ResNet50
from tensorflow.keras import layers, models
from tensorflow.keras.utils import to_categorical


In [2]:
(x_train, y_train), (x_test, y_test) = mnist.load_data()


11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [3]:
def preprocess_mnist(x, y):

    x = tf.image.resize(tf.expand_dims(x, axis=-1), (32, 32))
    x = tf.repeat(x, 3, axis=-1)

    x = tf.cast(x, tf.float32)
    x = tf.keras.applications.resnet50.preprocess_input(x)
    y = to_categorical(y, 10)

    return x, y


In [4]:
x_train_pp, y_train_pp = preprocess_mnist(x_train, y_train)
x_test_pp, y_test_pp = preprocess_mnist(x_test, y_test)


In [5]:
base_model = ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(32, 32, 3)
)


94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [6]:
base_model.trainable = False


In [7]:
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')
])


In [8]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)


In [9]:
model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)           │ (None, 1, 1, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 10)             │        20,490 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,608,202 (90.06 MB)

 Trainable params: 20,490 (80.04 KB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [10]:
history = model.fit(
    x_train_pp, y_train_pp,
    batch_size=128,
    epochs=5,
    validation_split=0.1,
    verbose=1
)


Epoch 1/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 230s 523ms/step - accuracy: 0.5869 - loss: 1.6517 - val_accuracy: 0.9115 - val_loss: 0.3187
Epoch 2/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 258s 513ms/step - accuracy: 0.8457 - loss: 0.4822 - val_accuracy: 0.9242 - val_loss: 0.2696
Epoch 3/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 225s 533ms/step - accuracy: 0.8558 - loss: 0.4440 - val_accuracy: 0.9245 - val_loss: 0.2645
Epoch 4/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 217s 515ms/step - accuracy: 0.8600 - loss: 0.4330 - val_accuracy: 0.9198 - val_loss: 0.2592
Epoch 5/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 237s 562ms/step - accuracy: 0.8641 - loss: 0.4184 - val_accuracy: 0.9280 - val_loss: 0.2386


In [11]:
test_loss, test_acc = model.evaluate(x_test_pp, y_test_pp, verbose=0)
print(f"Test accuracy: {test_acc:.4f}")


Test accuracy: 0.9229
